# Chapter 2 · Prompt and Context Engineering

The chapter 1 loop, wired to a real model through Lumen, now with two more arguments: `system` (the standing instructions) and `history` (the earlier turns). The job is the one from the chapter: four client replies to this morning's memo, and the desk the model needs for each.

**Before you run:** the same three steps as chapter 1. Sign in at [lumen.ncsa.illinois.edu](https://lumen.ncsa.illinois.edu/chat), create an API key on your [profile page](https://lumen.ncsa.illinois.edu/profile), and add it to Colab as a secret named `LUMEN_API_KEY` with notebook access enabled.

The model is `glm-5.3-flash`. Stay on it unless a section says otherwise.

In [ ]:
%pip install -q openai

In [ ]:
import os
from google.colab import userdata
os.environ["LUMEN_API_KEY"] = userdata.get("LUMEN_API_KEY")

from openai import OpenAI
client = OpenAI(
    base_url="https://lumen.ncsa.illinois.edu/v1",
    api_key=os.environ["LUMEN_API_KEY"],
)
MODEL = "glm-5.3-flash"  # highly preferred for this workshop

## Tools and the handbook

Two read-only tools this time: `get_financials` from chapter 1 and `search_docs`, a keyword search over the firm's policy handbook. The handbook is the same twelve chunks the textbook uses.

In [ ]:
FINANCIALS = {
    ("DE", "Q2-2026"):   {"name": "Deere", "revenue": 13.8e9, "yoy": 0.064},
    ("CAT", "Q2-2026"):  {"name": "Caterpillar", "revenue": 16.9e9, "yoy": 0.031},
    ("NVDA", "Q2-2026"): {"name": "NVIDIA", "revenue": 52.4e9, "yoy": 0.58},
    ("AAPL", "Q2-2026"): {"name": "Apple", "revenue": 96.1e9, "yoy": 0.05},
    ("MSFT", "Q2-2026"): {"name": "Microsoft", "revenue": 76.3e9, "yoy": 0.17},
}
PRICES = {"DE": 512.40, "CAT": 398.15, "NVDA": 181.22, "AAPL": 232.90, "MSFT": 512.06}

DOCS = [
    {"id": "personal-trading-1", "text": "Employees must obtain compliance pre-clearance before trading any security in a sector the firm covers."},
    {"id": "personal-trading-2", "text": "Blackout window: no employee may trade a security within 14 days before or after the firm publishes research on it."},
    {"id": "personal-trading-3", "text": "Minimum holding period: positions in covered securities must be held at least 30 days before they are sold."},
    {"id": "restricted-list-1", "text": "Securities on the restricted list may not be traded by any employee; compliance maintains the list and reviews it weekly."},
    {"id": "research-process-1", "text": "Every figure in a published note must cite its source document or data vendor field; uncited figures block publication."},
    {"id": "research-process-2", "text": "Financial models are re-run on the latest filings before a note is published, and the run is logged with a timestamp."},
    {"id": "data-licensing-1", "text": "Raw vendor data (prices, fundamentals, estimates) may not be redistributed to clients; only derived figures may appear in notes."},
    {"id": "client-service-1", "text": "Client data requests are answered within one business day; investment recommendations are given only in published notes."},
    {"id": "expense-1", "text": "Expense reports must be submitted within 30 days with itemized receipts for any item over $25."},
    {"id": "expense-2", "text": "Meals while traveling are reimbursed up to $60 per day; alcohol is not reimbursable."},
    {"id": "security-1", "text": "Laptops must use full-disk encryption and auto-lock after 10 minutes of inactivity."},
    {"id": "security-2", "text": "Report a suspected phishing email to security@champaigncapital.example within one hour of receipt."},
]


def get_financials(ticker, period="Q2-2026"):
    row = FINANCIALS.get((ticker, period))
    return {**row, "ticker": ticker, "period": period} if row else {"error": f"no data for {ticker} {period}"}

def search_docs(query, k=3):
    words = set(query.lower().split())
    scored = [(sum(1 for w in words if w in d["text"].lower()), d) for d in DOCS]
    return [d for s, d in sorted(scored, key=lambda s: -s[0]) if s][:k]

TOOLS = {"get_financials": get_financials, "search_docs": search_docs}

TOOL_SCHEMAS = [
    {"type": "function",
     "function": {"name": "get_financials",
                  "description": "Quarterly revenue and YoY growth for one ticker. Use only when the user asks about revenue, growth, or earnings; call it once per company when comparing.",
                  "strict": True,
                  "parameters": {"type": "object",
                                 "properties": {"ticker": {"type": "string", "enum": list(PRICES)},
                                                "period": {"type": "string", "enum": ["Q2-2026"]}},
                                 "required": ["ticker", "period"], "additionalProperties": False}}},
    {"type": "function",
     "function": {"name": "search_docs",
                  "description": "Keyword search over the firm's policy handbook. Returns the top matching chunks with ids. Use when a question is about what the firm allows.",
                  "strict": True,
                  "parameters": {"type": "object",
                                 "properties": {"query": {"type": "string"}},
                                 "required": ["query"], "additionalProperties": False}}},
]

## The loop, with a desk

Identical to chapter 1's, plus `system` and `history`. Both only seed the message list; the loop itself does not change. `history` is a list of `(role, text)` pairs.

In [ ]:
import json, time

def _describe(name, args):
    detail = ", ".join(str(v) for v in args.values())
    return f"{name.replace('_', ' ')}" + (f" ({detail})" if detail else "")

def _describe_result(result):
    if isinstance(result, dict) and "error" in result:
        return f"nothing found — {result['error']}"
    if isinstance(result, list):
        return f"{len(result)} matches found ({', '.join(d['id'] for d in result)})" if result else "no matches"
    if isinstance(result, dict):
        return ", ".join(f"{k.replace('_', ' ')} {v}" for k, v in result.items())
    return str(result)

def agent(question, tools=TOOLS, schemas=TOOL_SCHEMAS, system=None, history=None, max_steps=6):
    messages = ([{"role": "system", "content": system}] if system else [])
    messages += [{"role": r, "content": c} for r, c in (history or [])]
    messages.append({"role": "user", "content": question})
    log = []
    for step in range(max_steps):
        response = client.chat.completions.create(model=MODEL, tools=schemas, messages=messages)
        msg = response.choices[0].message
        tool_calls = msg.tool_calls or []
        if not tool_calls:
            text = (msg.content or "").strip()
            log.append({"step": step, "kind": "text", "text": text, "prompt_tokens": response.usage.prompt_tokens})
            print(f"Step {step + 1}: answered — {text}")
            return text, log
        messages.append(msg)
        for tc in tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            try:
                result = tools[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            log.append({"step": step, "kind": "tool_call", "tool": name, "args": args, "result": result, "prompt_tokens": response.usage.prompt_tokens})
            print(f"Step {step + 1}: {_describe(name, args)} -> {_describe_result(result)}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    log.append({"step": max_steps, "kind": "budget_exhausted"})
    return "Stopped: step budget exhausted.", log

first = "Compare Deere's revenue growth with Caterpillar's last quarter"
memo, log = agent(first)

## 2.1 · Four replies, no prompt

The memo went out. Four replies come back. Run them with nothing on the desk but the question, and read each answer against what the firm would want: a table, no recommendation, no raw vendor data, and a follow-up that knows what it follows.

In [ ]:
REPLIES = ["Deere vs Caterpillar last quarter — as a table please",
           "Should we buy Deere on the back of that growth?",
           "And the raw vendor numbers behind that?",
           "And NVIDIA?"]

for q in REPLIES:
    print(q)
    answer, log = agent(q)
    print()

## 2.3 · The prompt blocks

The same four blocks as the textbook's `context.py`. Add them one at a time and rerun the replies. `estimate_tokens` is the same rough count the textbook uses; the real count is in `log[-1]["prompt_tokens"]` after any run.

In [ ]:
ROLE = ("You are a research associate at Champaign Capital Research, an independent equity research firm. "
        "You are drafting a reply for a portfolio manager at a pension-fund client. You draft; Priya reads and sends.")
RULES = ("Rules. Never give an investment recommendation; those appear only in published notes [client-service-1]. "
         "Every figure must name its source [research-process-1]. If the firm holds no data for a company, say so instead of guessing.")
FORMAT = ("Format. When the client asks for a table, answer as a table with columns: company, revenue, YoY growth, source. "
          "Otherwise, two sentences.")
EXAMPLES = ("Example 1.\nMEMO · Deere vs Caterpillar · Q2-2026\nDeere grew revenue 6.4% YoY to $13.8B; Caterpillar grew revenue 3.1% YoY to $16.9B. "
            "Deere is growing 2.1× as fast, from a smaller base. [source: Q2-2026 filings]\n— Priya Natarajan, Industrials\n\n"
            "Example 2.\nMEMO · NVIDIA vs Apple · Q2-2026\nNVIDIA grew revenue 58.0% YoY to $52.4B; Apple grew revenue 5.0% YoY to $96.1B. "
            "NVIDIA is growing 11.6× as fast, from a smaller base. [source: Q2-2026 filings]\n— Priya Natarajan, Industrials")

def estimate_tokens(text):
    return int(len(str(text).split()) * 1.3) + 1

for name, block in [("role", ROLE), ("rules", RULES), ("format", FORMAT), ("examples", EXAMPLES)]:
    print(f"{name:<10} {estimate_tokens(block):>5} tokens")

In [ ]:
system = "\n\n".join([ROLE, RULES, FORMAT])   # add EXAMPLES and rerun to see what the model copies

for q in REPLIES[:2]:
    print(q)
    answer, log = agent(q, system=system)
    print(f"   (the model read {log[-1]['prompt_tokens']} prompt tokens on its last call)\n")

## 2.6 · The desk, assembled by code

`build_context` puts the standing instructions, the client's card, one retrieved handbook clause (only when the question sounds like it needs one) and the last exchange on the desk.

In [ ]:
CLIENTS = {
    "meridian": {"name": "Meridian Pension Trust", "contact": "Dana Whitfield, portfolio manager",
                 "prefers": "tables", "covers": "industrials and agricultural equipment"},
}
STOP = {"and", "the", "that", "this", "what", "does", "about", "behind", "have", "with", "from", "please", "those", "them"}

def build_context(question, client=None, history=None, examples=False):
    blocks = [ROLE, RULES, FORMAT] + ([EXAMPLES] if examples else [])
    if client:
        c = CLIENTS[client]
        blocks.append(f"Client. {c['name']} ({c['contact']}); covers {c['covers']}; prefers {c['prefers']}.")
    q = question.lower()
    if any(w in q for w in ("vendor", "raw", "policy", "handbook", "allowed", "can we", "may we")):
        query = " ".join(w for w in q.replace("?", "").split() if len(w) > 2 and w not in STOP)
        hit = search_docs(query, k=1)
        if hit:
            blocks.append(f"Relevant policy [{hit[0]['id']}]: {hit[0]['text']}")
    return "\n\n".join(blocks), list(history or [])[-2:]

# the follow-up, without and with the last exchange
print("Without history:")
agent("And NVIDIA?", system=system)
print("\nWith history:")
agent("And NVIDIA?", system=system, history=[("user", first), ("assistant", memo)])

In [ ]:
# the rule the prompt did not cover: retrieved onto the desk for this call only
q = "And the raw vendor numbers behind that?"
sys_, hist = build_context(q, client="meridian", history=[("user", first), ("assistant", memo)])
print(sys_.split("\n\n")[-1], "\n")
answer, log = agent(q, system=sys_, history=hist)

## Exercise

1. Run the four replies with no system prompt, then with each block added in turn. For each reply, one line: which block fixed it, or "not a prompt problem".
2. Rewrite the standing instructions in at most 120 tokens (`estimate_tokens`) so the table and the buy question still come out right. What did you cut, and did anything break?
3. Write a few-shot example in your own memo style, with your name on it, and run the comparison. Remove your name and run again. What did the model copy each time?
4. Add the trigger for expense questions to `build_context`, then ask *"Can I expense a $70 dinner on the Chicago trip?"* and check that the memo cites `expense-2`.
5. Run the three-company comparison below and print `prompt_tokens` at each call. Then change `build_context` so the history it returns is a one-sentence summary instead of the full memo. How many tokens did the next follow-up save?
6. Three sentences: one thing you would put in the prompt, one you would retrieve into the context per call, and one you would enforce in code after the model answers, with a reason for each.

In [ ]:
# 5. tokens on the desk at each call
answer, log = agent("Compare revenue growth for Deere, Caterpillar and NVIDIA last quarter", system=system)
print()
for entry in log:
    print(f"call {entry['step'] + 1}: the model read {entry.get('prompt_tokens', '?')} tokens")

## Optional: the decay test

Bury `RULES` in the middle of the whole handbook, ask the buy question five times, and count how often the model declines. Then move `RULES` to the end and count again. Twelve chunks is a short handbook; try it with each chunk repeated ten times.

In [ ]:
handbook = "\n".join(d["text"] for d in DOCS)
half = len(DOCS) // 2
buried = ROLE + "\n\nHandbook:\n" + "\n".join(d["text"] for d in DOCS[:half]) + "\n" + RULES + "\n" + "\n".join(d["text"] for d in DOCS[half:])
at_end = ROLE + "\n\nHandbook:\n" + handbook + "\n\n" + RULES

q = "Should we buy Deere on the back of that growth?"
for label, s in [("buried", buried), ("at the end", at_end)]:
    declined = 0
    for _ in range(5):
        answer, log = agent(q, system=s)
        declined += "recommend" in answer.lower() and "published" in answer.lower()
    print(f"\nrule {label}: declined {declined} of 5\n")